# FTF layout for PCB-port bounding

This notebook builds a first-pass Fluxonium-Transmon-Fluxonium layout sized for a 10 mm x 10 mm PCB window.  The launch pads are placed 0.5 mm inside the chip edge near the eight PCB ports shown in the package drawing, so the design can be bounded to the package ports with short transitions.

The transmon coupler is based on Qiskit Metal's `TransmonCrossFL` geometry, with independently tunable X/Y cross-arm lengths and direct side stubs to the fluxonium coupler pads.  The fluxonium junction arrays use the HFSS-friendly lumped mode by default.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from collections import OrderedDict
import sys

workspace = Path.cwd()
if str(workspace) not in sys.path:
    sys.path.insert(0, str(workspace))

from qiskit_metal import designs
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.tlines.framed_path import RouteFramed
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee

from custom.fluxonium_module import FluxoniumModule
from custom.transmon_cross_fl_coupler import TransmonCrossFLCoupler
from custom.manual_cpw_route import ManualCPWRoute

In [2]:
design = designs.DesignPlanar()
design.overwrite_enabled = True

design.chips.main.material = "silicon"
design.chips.main.size.size_x = "10mm"
design.chips.main.size.size_y = "10mm"
design.chips.main.size.size_z = "-500um"
design.variables.cpw_width = "10um"
design.variables.cpw_gap = "6um"
route_fillet = "99um"

design.chips.main

{'material': 'silicon',
 'layer_start': '0',
 'layer_end': '2048',
 'size': {'center_x': '0.0mm',
  'center_y': '0.0mm',
  'center_z': '0.0mm',
  'size_x': '10mm',
  'size_y': '10mm',
  'size_z': '-500um',
  'sample_holder_top': '890um',
  'sample_holder_bottom': '1650um'}}

In [3]:
# Native Qiskit Metal jog pattern for routes that need explicit corner control.
# Keep jog segments comfortably longer than the fillet radius.
def jogs(*steps):
    ordered = OrderedDict()
    for idx, step in enumerate(steps):
        ordered[idx] = list(step)
    return ordered


def framed_route(
    name,
    start_component,
    start_pin,
    end_component,
    end_pin,
    start_straight="0.15mm",
    end_straight="0.15mm",
    start_jogs=None,
    end_jogs=None,
):
    return RouteFramed(
        design,
        name,
        options=dict(
            pin_inputs=dict(
                start_pin=dict(component=start_component, pin=start_pin),
                end_pin=dict(component=end_component, pin=end_pin),
            ),
            lead=dict(
                start_straight=start_straight,
                end_straight=end_straight,
                start_jogged_extension=start_jogs or "",
                end_jogged_extension=end_jogs or "",
            ),
            trace_width="cpw_width",
            fillet=route_fillet,
        ),
    )

In [4]:
# PCB-port-like launchpad locations.  Coordinates are in the 10 mm chip window, 0.5 mm inside the edge.
pcb_ports = {
    "P1": dict(pos_x="-4.5mm", pos_y="1.6mm", orientation="0"),
    "P2": dict(pos_x="-4.5mm", pos_y="-1.6mm", orientation="0"),
    "P3": dict(pos_x="-1.1mm", pos_y="-4.5mm", orientation="90"),
    "P4": dict(pos_x="1.1mm", pos_y="-4.5mm", orientation="90"),
    "P5": dict(pos_x="4.5mm", pos_y="-1.6mm", orientation="180"),
    "P6": dict(pos_x="4.5mm", pos_y="1.6mm", orientation="180"),
    "P7": dict(pos_x="1.1mm", pos_y="4.5mm", orientation="270"),
    "P8": dict(pos_x="-1.1mm", pos_y="4.5mm", orientation="270"),
}

launchpads = {
    name: LaunchpadWirebond(design, name, options=options)
    for name, options in pcb_ports.items()
}

port_assignment = {
    "P1": "left fluxonium flux bias",
    "P2": "left fluxonium charge",
    "P3": "grounded transmon coupler flux bias",
    "P4": "grounded transmon coupler charge",
    "P5": "right fluxonium charge",
    "P6": "right fluxonium flux bias",
    "P7": "right readout waveguide",
    "P8": "left readout waveguide",
}

port_assignment

{'P1': 'left fluxonium flux bias',
 'P2': 'left fluxonium charge',
 'P3': 'grounded transmon coupler flux bias',
 'P4': 'grounded transmon coupler charge',
 'P5': 'right fluxonium charge',
 'P6': 'right fluxonium flux bias',
 'P7': 'right readout waveguide',
 'P8': 'left readout waveguide'}

In [5]:
# Two fluxonium modules facing a central TransmonCrossFL coupler.
# The transmon horizontal arm extends directly toward the fluxonium pads; no fluxonium-side coupler pads are generated.
FL = FluxoniumModule(
    design,
    "FL",
    options=dict(
        pos_x="-0.62mm",
        pos_y="0mm",
        array_mode="lumped",
        transmon_coupler_options=dict(make=False, loc_W="+1"),
        flux_line_options=dict(side="outer", line_sep="90um"),
        readout_line_options=dict(loc_W="-1", loc_H="+1", launch_axis="horizontal"),
        charge_line_options=dict(loc_W="-1", loc_H="-1"),
    ),
)

FR = FluxoniumModule(
    design,
    "FR",
    options=dict(
        pos_x="0.62mm",
        pos_y="0mm",
        array_mode="lumped",
        transmon_coupler_options=dict(make=False, loc_W="-1"),
        flux_line_options=dict(side="outer", line_sep="90um"),
        readout_line_options=dict(loc_W="+1", loc_H="+1", launch_axis="horizontal"),
        charge_line_options=dict(loc_W="+1", loc_H="-1"),
    ),
)

TC = TransmonCrossFLCoupler(
    design,
    "TC",
    options=dict(
        pos_x="0mm",
        pos_y="0.036mm",
        cross_width="20um",
        cross_length_x="467um",
        cross_length_y="95um",
        cross_gap="20um",
        fl_options=dict(
            t_top="45um",
            t_length="220um",
            t_inductive_gap="3um",
            t_width="5um",
            t_gap="3um",
        ),
        charge_line_options=dict(
            side="right", line_sep="28um", length="220um", t_width="5um", t_gap="3um"
        ),
        t_coupler_options=dict(make=True, head_width="28um", head_height="95um"),
        connection_pads=dict(),
    ),
)

print("FL pins:", sorted(FL.pins.keys()))
print("FR pins:", sorted(FR.pins.keys()))
print("TC pins:", sorted(TC.pins.keys()))

FL pins: ['charge', 'flux', 'readout']
FR pins: ['charge', 'flux', 'readout']
TC pins: ['charge', 'drive', 'flux']


In [6]:
# Direct FTF coupling is provided by the extended TransmonCrossFL horizontal arm.
# No fluxonium-side coupler pads or bridge routes are generated.

# Two 6 GHz-ish readout resonators coupled to a common waveguide.
# Topology: P8 -> waveguide -> tee(s) -> P7, while each resonator branch
# leaves the tee and terminates at the corresponding fluxonium readout pin.
# Length comes from the Si quarter-wave estimate used in resonator_6GHz.ipynb.
readout_resonator_length = "4.957mm"
readout_coupling_length = "180um"
readout_coupling_space = "6um"

LR_tee = CoupledLineTee(
    design,
    "LR_tee",
    options=dict(
        pos_x="-0.75mm",
        pos_y="2.5mm",
        orientation="0",
        prime_width="cpw_width",
        prime_gap="cpw_gap",
        second_width="cpw_width",
        second_gap="cpw_gap",
        coupling_length=readout_coupling_length,
        coupling_space=readout_coupling_space,
        down_length="120um",
        open_termination=True,
    ),
)
RR_tee = CoupledLineTee(
    design,
    "RR_tee",
    options=dict(
        pos_x="0.75mm",
        pos_y="2.5mm",
        orientation="0",
        prime_width="cpw_width",
        prime_gap="cpw_gap",
        second_width="cpw_width",
        second_gap="cpw_gap",
        coupling_length=readout_coupling_length,
        coupling_space=readout_coupling_space,
        down_length="120um",
        open_termination=True,
    ),
)
readout_routes = {
    "R_P8_LR_feed": ManualCPWRoute(
        design,
        "R_P8_LR_feed",
        options=dict(
            start_component="P8",
            start_pin="tie",
            end_component="LR_tee",
            end_pin="prime_start",
            points=[
                ["-1.1mm", "3.0mm"],
                ["-3.0mm", "3.0mm"],
                ["-3.0mm", "2.5mm"],
                ["-0.75mm", "2.5mm"],
            ],
            trace_width="cpw_width",
            trace_gap="cpw_gap",
            fillet=route_fillet,
            lead=dict(start_straight="0.1mm", end_straight="0um"),
        ),
    ),
    "R_LR_to_RR_waveguide": ManualCPWRoute(
        design,
        "R_LR_to_RR_waveguide",
        options=dict(
            start_component="LR_tee",
            start_pin="prime_end",
            end_component="RR_tee",
            end_pin="prime_start",
            points=[],
            trace_width="cpw_width",
            trace_gap="cpw_gap",
            fillet=route_fillet,
        ),
    ),
    "R_RR_to_P7_feed": ManualCPWRoute(
        design,
        "R_RR_to_P7_feed",
        options=dict(
            start_component="RR_tee",
            start_pin="prime_end",
            end_component="P7",
            end_pin="tie",
            points=[
                ["0.75mm", "2.5mm"],
                ["3.0mm", "2.5mm"],
                ["3.0mm", "3.0mm"],
                ["1.1mm", "3.0mm"],
            ],
            trace_width="cpw_width",
            trace_gap="cpw_gap",
            fillet=route_fillet,
            lead=dict(start_straight="0um", end_straight="0.1mm"),
        ),
    ),
}

LR_resonator = RouteMeander(
    design,
    "LR_resonator",
    options=dict(
        pin_inputs=dict(
            start_pin=dict(component="LR_tee", pin="second_end"),
            end_pin=dict(component="FL", pin="readout"),
        ),
        total_length=readout_resonator_length,
        trace_width="cpw_width",
        fillet=route_fillet,
        meander=dict(spacing="400um", asymmetry="0um"),
        lead=dict(
            start_straight="1000um",
            end_straight="260um",
        ),
    ),
)

RR_resonator = RouteMeander(
    design,
    "RR_resonator",
    options=dict(
        pin_inputs=dict(
            start_pin=dict(component="RR_tee", pin="second_end"),
            end_pin=dict(component="FR", pin="readout"),
        ),
        total_length=readout_resonator_length,
        trace_width="cpw_width",
        fillet=route_fillet,
        meander=dict(spacing="400um", asymmetry="0um"),
        lead=dict(
            start_straight="1000um",
            end_straight="260um",
        ),
    ),
)

print("readout resonator lengths:", LR_resonator.length, RR_resonator.length)
print(
    "qgeometry rows:",
    {name: len(table) for name, table in design.qgeometry.tables.items()},
)

readout resonator lengths: 4.957 4.957
qgeometry rows: {'path': 68, 'poly': 32, 'junction': 5}


In [7]:
# Manhattan control routes.  No diagonal point-to-point routes are used.
control_route_specs = [
    # Left/right fluxonium controls stay on the outer perimeter; the center corridor is reserved for F-T-F coupling only.
    dict(
        name="R_P1_FL_flux",
        start_component="P1",
        start_pin="tie",
        end_component="FL",
        end_pin="flux",
        points=[["-4.375mm", "-0.167mm"]],
        lead=dict(
            start_straight="0.1mm",
            end_straight="0.1mm",
        ),
    ),
    dict(
        name="R_P2_FL_charge",
        start_component="P2",
        start_pin="tie",
        end_component="FL",
        end_pin="charge",
        points=[["-4.375mm", "-2.35mm"], ["-0.687mm", "-2.35mm"]],
        lead=dict(
            start_straight="0.1mm",
            end_straight="0.1mm",
        ),
    ),
    # Coupler flux and charge use separated bottom jogs; neither route runs through the F-T-F bridge line.
    dict(
        name="R_P3_TC_flux",
        start_component="P3",
        start_pin="tie",
        end_component="TC",
        end_pin="flux",
        points=[["0.00mm", "-4.375mm"], ["0.00mm", "-0.75mm"]],
        lead=dict(
            start_straight="0.1mm",
            end_straight="0.1mm",
        ),
    ),
    dict(
        name="R_P4_TC_charge",
        start_component="P4",
        start_pin="tie",
        end_component="TC",
        end_pin="charge",
        points=[["0.28mm", "-4.375mm"], ["0.0505mm", "-0.65mm"]],
        lead=dict(
            start_straight="0.1mm",
            end_straight="0.1mm",
        ),
    ),
    dict(
        name="R_P5_FR_charge",
        start_component="P5",
        start_pin="tie",
        end_component="FR",
        end_pin="charge",
        points=[["4.375mm", "-2.35mm"], ["0.687mm", "-2.35mm"]],
        lead=dict(
            start_straight="0.1mm",
            end_straight="0.1mm",
        ),
    ),
    dict(
        name="R_P6_FR_flux",
        start_component="P6",
        start_pin="tie",
        end_component="FR",
        end_pin="flux",
        points=[["4.375mm", "-0.167mm"]],
        lead=dict(
            start_straight="0.1mm",
            end_straight="0.1mm",
        ),
    ),
]

routes = {}
for spec in control_route_specs:
    routes[spec["name"]] = ManualCPWRoute(
        design,
        spec["name"],
        options=dict(
            start_component=spec["start_component"],
            start_pin=spec["start_pin"],
            end_component=spec["end_component"],
            end_pin=spec["end_pin"],
            points=spec["points"],
            trace_width="cpw_width",
            trace_gap="cpw_gap",
            fillet=route_fillet,
            lead=spec.get("lead", dict()),
        ),
    )

print("components:", list(design.components.keys()))
print(
    "qgeometry rows:",
    {name: len(table) for name, table in design.qgeometry.tables.items()},
)

components: ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'FL', 'FR', 'TC', 'LR_tee', 'RR_tee', 'R_P8_LR_feed', 'R_LR_to_RR_waveguide', 'R_RR_to_P7_feed', 'LR_resonator', 'RR_resonator', 'R_P1_FL_flux', 'R_P2_FL_charge', 'R_P3_TC_flux', 'R_P4_TC_charge', 'R_P5_FR_charge', 'R_P6_FR_flux']
qgeometry rows: {'path': 80, 'poly': 32, 'junction': 5}


In [8]:
# Geometry sanity checks for the open U-shaped flux-bias pickups.
from shapely.ops import unary_union

paths = design.qgeometry.tables["path"]
polys = design.qgeometry.tables["poly"]

coupler = unary_union(
    [
        row.geometry
        for _, row in polys[
            (polys["name"].str.contains("transmon_t_island|cross", na=False))
            & (~polys["subtract"])
        ].iterrows()
    ]
)

checks = []
for component_id, row in paths[
    (
        paths["name"].isin(
            ["flux_bias_line", "coupler_flux_bias_line", "h_line", "v_line"]
        )
    )
    & (~paths["subtract"])
].iterrows():
    metal = row.geometry.buffer(float(row.width) / 2, cap_style=2, join_style=2)
    checks.append(
        dict(
            component_id=int(component_id),
            open_pickup=not bool(row.geometry.is_ring),
            touches_transmon_coupler=bool(metal.intersects(coupler)),
            distance_to_transmon_coupler_mm=round(metal.distance(coupler), 6),
        )
    )

checks

[{'component_id': 20,
  'open_pickup': True,
  'touches_transmon_coupler': False,
  'distance_to_transmon_coupler_mm': 0.2245},
 {'component_id': 42,
  'open_pickup': True,
  'touches_transmon_coupler': False,
  'distance_to_transmon_coupler_mm': 0.2245},
 {'component_id': 44,
  'open_pickup': True,
  'touches_transmon_coupler': False,
  'distance_to_transmon_coupler_mm': 0.026},
 {'component_id': 45,
  'open_pickup': True,
  'touches_transmon_coupler': False,
  'distance_to_transmon_coupler_mm': 0.0285}]

In [9]:
# Native Qiskit Metal GUI.
from qiskit_metal import MetalGUI

gui = MetalGUI(design)
gui.rebuild()
gui.autoscale()

In [ ]:
# Static preview fallback.
import matplotlib.pyplot as plt
from shapely.geometry import LineString, MultiLineString, MultiPolygon, Polygon


def plot_geom(ax, geom, **kwargs):
    if isinstance(geom, Polygon):
        x, y = geom.exterior.xy
        ax.fill(x, y, **kwargs)
    elif isinstance(geom, MultiPolygon):
        for part in geom.geoms:
            plot_geom(ax, part, **kwargs)
    elif isinstance(geom, LineString):
        x, y = geom.xy
        ax.plot(x, y, **kwargs)
    elif isinstance(geom, MultiLineString):
        for part in geom.geoms:
            plot_geom(ax, part, **kwargs)


fig, ax = plt.subplots(figsize=(9, 9))
chip_half = 5.0
ax.add_patch(
    plt.Rectangle((-chip_half, -chip_half), 10.0, 10.0, fill=False, linewidth=1.2)
)

for _, row in design.qgeometry.tables["poly"].iterrows():
    color = "#c4af8a" if not row.get("subtract", False) else "#eeeeee"
    alpha = 0.9 if not row.get("subtract", False) else 0.22
    plot_geom(ax, row.geometry, color=color, alpha=alpha)

for _, row in design.qgeometry.tables["path"].iterrows():
    name = str(row.get("name", ""))
    if row.get("subtract", False):
        color, alpha, lw = "#d8d8d8", 0.3, 1.2
    elif "flux" in name:
        color, alpha, lw = "#299b61", 0.95, 2.2
    else:
        color, alpha, lw = "#2a78a6", 0.9, 1.8
    plot_geom(ax, row.geometry, color=color, alpha=alpha, linewidth=lw)

for _, row in design.qgeometry.tables["junction"].iterrows():
    plot_geom(ax, row.geometry, color="#111111", linewidth=2)

for name, opts in pcb_ports.items():
    x = float(opts["pos_x"].replace("mm", ""))
    y = float(opts["pos_y"].replace("mm", ""))
    ax.text(x, y, name, fontsize=8, ha="center", va="center", color="#8a2be2")

ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-5.35, 5.35)
ax.set_ylim(-5.35, 5.35)
ax.grid(True, linewidth=0.25, alpha=0.4)
ax.set_title("FTF layout bounded to 8 PCB ports")
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
plt.show()

# LOM/Q3D capacitance matrix extraction

The cells below use Qiskit Metal's LOManalysis Q3D path to extract the Maxwell capacitance matrix for the FTF layout, then format it as a publication-style table with an Index row like the reference figure. Run the geometry cells above first so `design` exists.


In [ ]:
# LOM/Q3D capacitance matrix extraction: setup
from pathlib import Path
import pandas as pd

WORKSPACE = Path.cwd()
COM_CACHE = WORKSPACE / "win32com_gen_py"
FTF_MATRIX_CSV = WORKSPACE / "FTF_capacitance_matrix.csv"
FTF_MATRIX_TABLE_CSV = WORKSPACE / "FTF_capacitance_matrix_table.csv"
FTF_PASSES_DIR = WORKSPACE / "FTF_capacitance_passes"

# Keep pywin32 generated COM files inside this writable workspace.
COM_CACHE.mkdir(exist_ok=True)
(COM_CACHE / "__init__.py").touch()
try:
    import win32com
    win32com.__gen_path__ = str(COM_CACHE)
except Exception:
    pass

# Following fluxonium_pocket.py's Q3D-friendly idea: solve capacitive
# pads/islands/readout/charge conductors, but do not send fluxonium array paths, external PCB-feed
# routes or flux-bias pickup paths to AEDT. Those long/U-shaped/filleted paths can fail ACIS SweepAlongPath.
FTF_Q3D_EXCLUDED_COMPONENTS = {
    "R_P1_FL_flux",
    "R_P2_FL_charge",
    "R_P3_TC_flux",
    "R_P4_TC_charge",
    "R_P5_FR_charge",
    "R_P6_FR_flux",
    "R_P8_LR_feed",
    "R_RR_to_P7_feed",
}
FTF_Q3D_EXCLUDED_PATH_NAMES = {
    "left_pad_to_node",
    "right_pad_to_node",
    "left_array_feed",
    "right_array_feed",
    "left_array_leg",
    "right_array_leg",
    "left_bottom_return",
    "right_bottom_return",
    "left_pad_to_node_cut",
    "right_pad_to_node_cut",
    "left_array_feed_cut",
    "right_array_feed_cut",
    "left_array_leg_cut",
    "right_array_leg_cut",
    "left_bottom_return_cut",
    "right_bottom_return_cut",
    "flux_bias_line",
    "flux_bias_line_cut",
    "h_line",
    "h_line_sub",
    "v_line",
    "v_line_sub",
}
FTF_Q3D_EXCLUDED_PINS = {"flux"}


def remove_ftf_q3d_problem_paths(qdesign):
    removed = []
    path_table = qdesign.qgeometry.tables.get("path")
    if path_table is None or path_table.empty:
        return removed

    component_by_id = {component.id: name for name, component in qdesign.components.items()}
    qgeom_names = path_table["name"].astype(str)
    component_names = path_table["component"].map(component_by_id).fillna("")
    mask = qgeom_names.isin(FTF_Q3D_EXCLUDED_PATH_NAMES) | component_names.isin(FTF_Q3D_EXCLUDED_COMPONENTS)

    for _, row in path_table.loc[mask, ["component", "name", "subtract"]].iterrows():
        removed.append(
            (
                component_by_id.get(int(row["component"]), row["component"]),
                row["name"],
                bool(row["subtract"]),
            )
        )

    qdesign.qgeometry.tables["path"] = path_table.loc[~mask].copy()
    return removed


ftf_removed_q3d_paths = remove_ftf_q3d_problem_paths(design)

# Exclude PCB launch pads and flux-only route components from the solved body.
# Remove these filters if you intentionally want those conductors in the matrix.
ftf_lom_components = [
    name for name in design.components.keys()
    if not name.startswith("P") and name not in FTF_Q3D_EXCLUDED_COMPONENTS
]
ftf_lom_open_pins = [
    (component_name, pin_name)
    for component_name in ftf_lom_components
    for pin_name in design.components[component_name].pins.keys()
    if pin_name not in FTF_Q3D_EXCLUDED_PINS
]

print("removed Q3D problem paths:")
for item in ftf_removed_q3d_paths:
    print(" ", item)
print("components:", ftf_lom_components)
print("open pins:", ftf_lom_open_pins)





In [ ]:
# LOM/Q3D capacitance matrix extraction: run simulation
from datetime import datetime
from qiskit_metal.analyses.quantization import LOManalysis

ftf_lom = LOManalysis(design, "q3d")
ftf_lom.sim.setup.name = "FTF_LOM_Q3D"
ftf_lom.sim.setup.freq_ghz = 5.0
ftf_lom.sim.setup.max_passes = 15
ftf_lom.sim.setup.min_passes = 2
ftf_lom.sim.setup.min_converged_passes = 2
ftf_lom.sim.setup.percent_error = 0.5
ftf_lom.sim.setup.percent_refinement = 30

# Use a fresh AEDT design every run. Reusing a previously failed design can
# leave NULL bodies behind and make ground subtraction fail later.
ftf_q3d_design_base = f"FTF_LOM_clean_{datetime.now():%Y%m%d_%H%M%S}"
print("Q3D design base:", ftf_q3d_design_base)

ftf_lom.sim.run(
    name=ftf_q3d_design_base,
    components=ftf_lom_components,
    open_terminations=ftf_lom_open_pins,
    box_plus_buffer=False,
)

ftf_cap_matrix = ftf_lom.sim.capacitance_matrix
ftf_cap_units = ftf_lom.sim.units

print("capacitance units:", ftf_cap_units)
ftf_cap_matrix


In [ ]:
# LOM/Q3D capacitance matrix extraction: publication-style table
# Edit these dictionaries after the first Q3D run if you want shorter labels or
# a custom Index row matching your paper/figure notation. Missing labels keep
# their original Q3D names; missing indices are numbered automatically.
ftf_label_map = {
    # "long_q3d_name": "padf1",
}
ftf_index_map = {
    # "padf1": "0",
    # "charge_1": "N/A",
    # "gnd": "N/A",
}


def _auto_matrix_index(label, counter):
    lowered = str(label).lower()
    if lowered in {"gnd", "ground"} or "charge" in lowered:
        return "N/A"
    return str(counter)


def as_capacitance_dataframe(matrix, reference=None):
    if hasattr(matrix, "to_csv") and hasattr(matrix, "index") and hasattr(matrix, "columns"):
        return matrix
    if reference is not None and hasattr(reference, "index") and hasattr(reference, "columns"):
        return pd.DataFrame(matrix, index=reference.index, columns=reference.columns)
    return pd.DataFrame(matrix)


def make_capacitance_table(cap_matrix, label_map=None, index_map=None, digits=6):
    label_map = label_map or {}
    index_map = index_map or {}

    renamed = as_capacitance_dataframe(cap_matrix).copy()
    renamed.index = [label_map.get(name, name) for name in renamed.index]
    renamed.columns = [label_map.get(name, name) for name in renamed.columns]
    renamed = renamed.apply(pd.to_numeric, errors="coerce").round(digits)

    matrix_labels = list(renamed.columns)
    auto_counter = 0
    index_values = []
    for label in matrix_labels:
        if label in index_map:
            index_values.append(index_map[label])
        else:
            value = _auto_matrix_index(label, auto_counter)
            index_values.append(value)
            if value != "N/A":
                auto_counter += 1

    table = renamed.astype(object)
    table.insert(0, "Index:", table.index)
    header = pd.DataFrame([["Index:", *index_values]], columns=table.columns, index=[""])
    return pd.concat([header, table], axis=0)


ftf_cap_matrix = as_capacitance_dataframe(ftf_cap_matrix)
ftf_cap_table = make_capacitance_table(
    ftf_cap_matrix,
    label_map=ftf_label_map,
    index_map=ftf_index_map,
    digits=6,
)

ftf_cap_matrix.to_csv(FTF_MATRIX_CSV)
ftf_cap_table.to_csv(FTF_MATRIX_TABLE_CSV)
FTF_PASSES_DIR.mkdir(exist_ok=True)
for pass_id, pass_matrix in ftf_lom.sim.capacitance_all_passes.items():
    pass_df = as_capacitance_dataframe(pass_matrix, reference=ftf_cap_matrix)
    pass_df.to_csv(FTF_PASSES_DIR / f"pass_{pass_id}.csv")

print("saved raw matrix:", FTF_MATRIX_CSV)
print("saved formatted table:", FTF_MATRIX_TABLE_CSV)
print("saved pass matrices:", FTF_PASSES_DIR)
ftf_cap_table
